In [1]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [3]:
load_dotenv(override=True)
google_api_key = os.getenv('GOOGLE_API_KEY')

if google_api_key:
    print(f"Google API 키가 존재하며 {google_api_key[:8]}로 시작합니다")
else:
    print("Google API 키가 설정되어 있지 않습니다")


Google API 키가 존재하며 AQ.Ab8RN로 시작합니다


In [4]:
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
MODEL = "gemini-3.1-flash-lite"
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)

In [6]:
system_message = """
당신은 FlightAI라는 항공사의 도움이 되는 어시스턴트입니다.
짧고 정중한 답변을 하며, 1문장을 넘기지 마세요.
항상 정확하게 답하세요. 답을 모른다면, 모른다고 말하세요.
"""

In [ ]:
def chat(message, history):
    history = [{"role" : h["role"], "content" : h["content"]} for h in history]
    messages = [{"role" : "system", "content" : system_message}] + history + [{"role":"user", "content":message}]
    response = gemini.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content


gr.ChatInterface(fn=chat, type="messages").launch()

# 도구 (Tools)

도구는 프론티어 LLM이 제공하는 대단히 강력한 기능입니다.

도구를 사용하면 함수를 작성하고, LLM이 응답의 일부로 그 함수를 호출하게 할 수 있습니다.

In [9]:
# 먼저 유용한 함수를 하나 만들어봅시다

ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}

def get_ticket_price(destination_city):
    print(f"{destination_city} 도시에 대해 도구가 호출되었습니다")
    price = ticket_prices.get(destination_city.lower(), "알 수 없는 티켓 가격")
    return f"{destination_city}로 가는 티켓 가격은 {price}입니다"

In [10]:
get_ticket_price("london")

london 도시에 대해 도구가 호출되었습니다


'london로 가는 티켓 가격은 $799입니다'

In [ ]:
# 우리 함수를 설명하기 위해 요구되는 특정한 딕셔너리 구조가 있습니다:

price_function = {
    "name" : "get_ticket_price",
    "description" : "목적지 도시로 가는 왕복 티켓의 가격을 가져옵니다.",
    "parameters" : {
        "type" : "object",
        "properties" : {
            "destination_city" : {
                "type" : "string",
                "description" : "고객이 여행하고자 하는 도시",
            },
        },
        "required" : ["destination_city"],
        "additionalProperties" : False
    }
}

In [12]:
# 그리고 이것이 도구 목록에 포함됩니다.

tools = [{"type" : "function", "function": price_function}]

tools

[{'type': 'function',
  'function': {'name': 'get_ticket_price',
   'description': '목적지 도시로 가는 왕복 티켓의 가격을 가져옵니다.',
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
      'description': '고객이 여행하고자 하는 도시'}},
    'required': ['destination_city'],
    'additionalProperties': False}}}]

## OpenAI가 우리 도구를 사용하게 하기

In [21]:
# handle_tool_call 함수를 작성해야 합니다:

def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get("destination_city")
            price_details = get_ticket_price(city)
            responses.append({
                "role" : "tool",
                "content" : price_details,
                "tool_call_id" : tool_call.id
            })
    return responses

In [ ]:
# 질문 : London행 가격을 확인하세요. 그리고 1000달러 미만이라면 Paris를 확인해 보세요
# 한번밖에 툴을 호출하지 않음

def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = gemini.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responsees = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responsees)

        for m in messages:
            print(m)

        response = gemini.chat.completions.create(model=MODEL, messages=messages)

    return response.choices[0].message.content


In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


London 도시에 대해 도구가 호출되었습니다
Paris 도시에 대해 도구가 호출되었습니다


In [26]:
# 툴을 반복해서 계속 돌려보고 싶다면 
# 질문 : London행 가격을 확인하세요. 그리고 1000달러 미만이라면 Paris를 확인해 보세요

def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = gemini.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = gemini.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    return response.choices[0].message.content